# 24.6 设计 ETA 预测系统 / Design an ETA Prediction System (Uber / DoorDash / 高德)

**中文**:ETA(预计到达时间)是出行/外卖/物流产品的核心——你打开 Uber 看到的"8 分钟后到"、DoorDash 的"30 分钟送达"、导航的"预计 45 分钟",背后都是一个 ML 系统。它和前面的题(推荐/搜索/广告/欺诈)最大的不同:那些都是**分类/排序**问题,而 ETA 是一道**回归**问题。但它有一个决定性的、极其能体现深度的洞察:**ETA 不应该是一个"点预测",而应该是一个"可靠的承诺"**。因为 ETA 的误差是**不对称的**——迟到 10 分钟(用户在门口干等、外卖凉了)比早到 10 分钟(用户还没准备好)糟糕得多。所以你不该预测"最可能的到达时间(中位数 P50)"——那有一半时间会迟到;而应该预测一个**留了缓冲的、可靠的时间(如 P90 分位数)**,让 90% 的情况都能准时。实现这个的技术是**分位数回归(quantile regression)**。本节从零演示 P50 vs P90 的天壤之别,再讲清 ETA 系统设计的完整框架。
**English**: ETA (estimated time of arrival) is the core of ride-hailing/delivery/logistics products — the "8 minutes away" on Uber, "30-minute delivery" on DoorDash, "estimated 45 min" in navigation are all backed by an ML system. Its biggest difference from prior questions (recommendation/search/ads/fraud): those are **classification/ranking** problems, while ETA is a **regression** problem. But it has a decisive, depth-revealing insight: **ETA shouldn't be a "point prediction" but a "reliable promise."** Because ETA errors are **asymmetric** — arriving 10 minutes late (the user waits at the door, the food gets cold) is far worse than arriving 10 minutes early (the user isn't ready yet). So you shouldn't predict "the most likely arrival time (median P50)" — that's late half the time; instead predict a **buffered, reliable time (like the P90 quantile)** so 90% of cases arrive on time. The technique is **quantile regression**. This section demonstrates the vast difference between P50 and P90 from scratch, then clarifies the complete ETA-system-design framework.

---

**中文**:**ETA 的核心洞察:预测"分布"而非"点",因为误差不对称**:
**English**: **ETA's core insight: predict a "distribution" not a "point," because errors are asymmetric**:
- **中文**:**为什么 P50(中位数)不够**:P50 是"最可能的时间"——但根据定义,实际时间有**一半会超过它**。如果你告诉用户"8 分钟到",而这是 P50,那有 50% 的概率你迟到了。对出行/外卖,迟到的体验伤害远大于早到,所以 P50 是个糟糕的承诺。
  **Why P50 (median) isn't enough**: P50 is "the most likely time" — but by definition, the actual time **exceeds it half the time**. If you tell the user "8 minutes" and that's P50, there's a 50% chance you're late. For ride-hailing/delivery, late experience hurts far more than early, so P50 is a poor promise.
- **中文**:**用分位数回归给"可靠承诺"**:预测 **P90 分位数**——"90% 的情况下会在这个时间之前到"。这天然留了缓冲,让绝大多数用户体验到"准时或提前"。实现靠**分位数回归(pinball / 分位数损失)**:损失函数对"低估"(预测早了、实际迟了)的惩罚是对"高估"的 9 倍(α=0.9),模型就学会预测一个偏保守的、90% 覆盖的时间。
  **Use quantile regression for a "reliable promise"**: predict the **P90 quantile** — "90% of the time it arrives before this." This naturally leaves a buffer so the vast majority experience "on time or early." Implemented via **quantile regression (pinball / quantile loss)**: the loss penalizes "underestimation" (predicted early, actually late) 9x more than "overestimation" (α=0.9), so the model learns a conservative, 90%-covering time.
- **中文**:**权衡**:P90 比 P50 更"保守"(报的时间更长),准时率高但承诺不够"激进"。产品要在**准时率(可靠性)** 和 **承诺的吸引力(报得越短越吸引人下单)** 之间权衡——通常选一个高分位数(P80~P90),甚至给出一个**区间**("25-35 分钟")。
  **Tradeoff**: P90 is more "conservative" than P50 (a longer quoted time), with higher on-time rate but a less "aggressive" promise. The product must trade off **on-time rate (reliability)** against **promise attractiveness (shorter quotes attract more orders)** — usually choosing a high quantile (P80–P90), or even giving a **range** ("25–35 minutes").

> 💡 **面试速查 / Interview cheat-sheet（★★★ ETA/回归系统设计, 高频）**
> **中文**:**ETA=回归问题, 但核心是预测分布/可靠承诺而非点**。**关键洞察**:误差**不对称**(迟到比早到伤害大)→不该预测 P50 中位数(一半会迟到), 而用**分位数回归**预测 **P90**(90% 准时)→留缓冲。损失=**分位数/pinball loss**(α=0.9 时低估惩罚 9 倍), 不是 MSE。**权衡**:准时率(高分位数)vs 承诺吸引力(短时间), 常给区间。**架构**:①**路段/图模型**——把路线拆成路段, 预测每段耗时再求和(或图神经网络), 而非端到端一个数;②**实时特征**——当前交通(流式)、天气、时段、历史;③外卖 ETA=接单+备餐+取餐+配送多段, 每段单独建模。**模型**:GBDT(强基线)、路段级 + 图网络、深度模型(时空)。**特征**:距离、实时路况、时段/星期、天气、历史该路段耗时、司机/餐厅状态、订单排队。**关键难题**:①**实时路况**(流式更新)②**长尾/罕见事件**(事故、暴雨→ETA 暴涨, 难预测)③**分段误差累积**④**反馈回路**(你的 ETA 影响用户是否下单/司机路线→影响真实到达)⑤新区域冷启动。**指标**:MAE/MAPE + **分位数覆盖率/准时率** + 在线(实际 vs 预测、用户满意)。面试金句:*"ETA 是回归但关键是误差不对称(迟到比早到伤), 所以用分位数回归预测 P90 可靠承诺(90%准时)而非 P50 中位数(一半迟到), 损失用 pinball 不是 MSE; 架构上把路线拆成路段分别预测再求和(或图网络)、外卖分接单/备餐/配送多段; 实时特征是路况(流式), 难点是长尾事件、分段误差累积、反馈回路; 权衡准时率与承诺吸引力。"*
> **English**: **ETA = regression, but the core is predicting a distribution / reliable promise, not a point**. **Key insight**: errors are **asymmetric** (late hurts more than early) → don't predict the P50 median (late half the time) but use **quantile regression** to predict **P90** (90% on time) → leave a buffer. Loss = **quantile/pinball loss** (at α=0.9, underestimation penalized 9x), not MSE. **Tradeoff**: on-time rate (high quantile) vs promise attractiveness (short time), often give a range. **Architecture**: ① **segment/graph model** — split the route into road segments, predict each segment's time and sum (or a graph neural network), rather than one end-to-end number; ② **real-time features** — current traffic (streaming), weather, time-of-day, history; ③ delivery ETA = order-accept + prep + pickup + delivery multiple stages, each modeled separately. **Models**: GBDT (strong baseline), segment-level + graph nets, deep spatio-temporal models. **Features**: distance, real-time traffic, time-of-day/day-of-week, weather, historical segment times, driver/restaurant state, order queue. **Key challenges**: ① **real-time traffic** (streaming updates) ② **long-tail/rare events** (accidents, storms → ETA spikes, hard to predict) ③ **segment error accumulation** ④ **feedback loop** (your ETA affects whether users order / driver routes → affects actual arrival) ⑤ new-region cold start. **Metrics**: MAE/MAPE + **quantile coverage/on-time rate** + online (actual vs predicted, user satisfaction). Interview line: *"ETA is regression but the key is asymmetric error (late hurts more than early), so use quantile regression to predict a P90 reliable promise (90% on time) not the P50 median (late half the time), with pinball loss not MSE; architecturally split the route into segments predicted and summed (or a graph net), delivery into order-accept/prep/delivery stages; real-time features are traffic (streaming), challenges are long-tail events, segment error accumulation, feedback loops; trade off on-time rate vs promise attractiveness."*


In [ ]:

# ============================================================
# 核心:P50(点预测)vs P90(可靠承诺)—— 分位数回归 / core: P50 (point) vs P90 (reliable promise) — quantile regression
# 中文:造一个 ETA 数据(耗时右偏, 因为交通/意外制造长尾)。对比:预测中位数 P50 vs 预测 P90 分位数。
#      看 P50 有一半时间迟到, 而 P90 留了缓冲 90% 准时。这就是为什么 ETA 用分位数回归而非普通回归。
# English: build ETA data (right-skewed times, since traffic/incidents create long tails). Compare predicting the
#      median P50 vs the P90 quantile. P50 is late half the time; P90 leaves a buffer, 90% on time. Hence quantile regression.
# ============================================================
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
np.random.seed(0)
N=8000
X=np.random.rand(N,4)                                       # 特征:距离/交通/时段/备餐 / distance, traffic, hour, prep
base=5 + 8*X[:,0] + 4*X[:,1] + 2*X[:,2]                     # 基础耗时 / base travel time
noise=np.random.exponential(3,N)*(0.5+X[:,1])              # 右偏噪声(交通制造长尾)/ right-skewed noise (traffic → long tail)
y=base+noise
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,random_state=0)
# P50:预测中位数(pinball loss, α=0.5)/ P50: predict the median
m50=GradientBoostingRegressor(loss="quantile",alpha=0.5,n_estimators=200,max_depth=3,random_state=0).fit(Xtr,ytr)
# P90:预测 90% 分位数(pinball loss α=0.9, 低估惩罚 9 倍→偏保守留缓冲)/ P90: predict the 90th percentile
m90=GradientBoostingRegressor(loss="quantile",alpha=0.9,n_estimators=200,max_depth=3,random_state=0).fit(Xtr,ytr)
p50=m50.predict(Xte); p90=m90.predict(Xte)
cover50=(yte<=p50).mean()                                   # 实际在预测时间之前到达的比例 / fraction arriving by the estimate
cover90=(yte<=p90).mean()
print("误差不对称: 迟到(实际>预测)比早到(实际<预测)对用户伤害大得多\n")
print(f"P50(中位数)ETA: 实际按时到达比例 = {cover50:.1%}  ← 约一半时间迟到! 作为承诺很糟")
print(f"P90(可靠)ETA:   实际按时到达比例 = {cover90:.1%}  ← 90% 时间准时, 才是可靠的'预计送达'")
print(f"\nP90 比 P50 平均多报 {(p90-p50).mean():.1f} 分钟(为可靠性预留的缓冲)")
print("→ ETA 不是点预测: 用分位数回归(pinball loss)预测 P90, 给用户可靠承诺, 而非一半会迟到的 P50")


In [ ]:

# ============================================================
# 可视化:P50 vs P90 覆盖率 + ETA 架构 / P50 vs P90 coverage + ETA architecture
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 准时率对比 + 误差分布 / on-time rate + error distribution
ax[0].bar(["P50\n(中位数点预测)","P90\n(分位数回归)"],[cover50*100,cover90*100],color=["#C44E52","#55A868"])
for i,v in enumerate([cover50*100,cover90*100]): ax[0].text(i,v+1,f"{v:.0f}%",ha="center",fontsize=12,weight="bold")
ax[0].axhline(90,ls="--",color="gray",alpha=0.6,label="90% 可靠目标")
ax[0].set_ylabel("实际按时到达比例 %"); ax[0].set_title("P50 一半迟到, P90 才是可靠承诺"); ax[0].legend(fontsize=8); ax[0].set_ylim(0,105)
# ② ETA 系统架构(分段)/ ETA architecture (segmented)
ax[1].axis("off"); ax[1].set_title("ETA 系统:分段预测 + 实时特征",fontsize=12,weight="bold")
ax[1].text(0.5,0.9,"外卖 ETA = 多段之和(每段单独建模)",ha="center",fontsize=9,weight="bold",transform=ax[1].transAxes)
segs=["① 接单等待","② 餐厅备餐","③ 骑手取餐","④ 路上配送(分路段/图网络)"]
for i,s in enumerate(segs):
    ax[1].add_patch(plt.Rectangle((0.08,0.72-i*0.13),0.84,0.1,fc="#4C72B0",alpha=0.25,ec="#4C72B0",transform=ax[1].transAxes))
    ax[1].text(0.5,0.77-i*0.13,s,ha="center",va="center",fontsize=8.5,transform=ax[1].transAxes)
ax[1].text(0.5,0.14,"实时特征: 当前路况(流式)/天气/时段/餐厅排队/骑手状态",ha="center",fontsize=8,transform=ax[1].transAxes)
ax[1].text(0.5,0.04,"输出: P90 分位数(可靠)或区间(25-35分钟)",ha="center",fontsize=8,style="italic",color="#55A868",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/sd06_viz.png",dpi=80); plt.show()
print("左:P50 只有 ~50% 准时(点预测的陷阱), P90 分位数回归 ~90% 准时; 右:ETA 分段建模+实时路况特征")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **ETA 系统最能体现深度的一点:回归问题不一定要预测"均值/中位数"**:大多数人一听"预测到达时间"就想到普通回归(最小化 MSE,预测期望值)。但 ETA 有一个决定性的业务事实——**误差是不对称的**:迟到 10 分钟(用户在门口焦虑等待、外卖凉了、错过航班)的痛苦,远大于早到 10 分钟。这个不对称性直接推翻了"预测中位数"的默认做法:我们的 demo 清清楚楚地显示,P50(中位数)预测只有约 50% 的准时率——因为按定义,实际时间有一半会超过中位数。正确的做法是**预测一个高分位数(P90)作为"可靠承诺"**,用**分位数回归(pinball loss)** 实现——它对低估的惩罚远大于高估,逼模型学出一个留了缓冲的时间。**"ETA 要预测分布/分位数而非点,因为误差不对称"是这道题最有深度的洞察,也是区分"会调回归模型"和"懂 ETA 业务"的分水岭。**
2. **ETA 的架构智慧:分段建模而非端到端一个数**:一个新手可能直接"输入所有特征,输出一个 ETA 数字"。但成熟的 ETA 系统会**把问题拆解**:外卖 ETA = 接单等待 + 餐厅备餐 + 骑手取餐 + 路上配送,每一段单独建模、各有各的特征和不确定性,最后求和。路上配送这一段还会进一步拆成**路段(road segments)**——把路线切成一段段路,预测每段的通行时间(基于该路段的实时路况和历史),再加起来;更先进的用**图神经网络**(路网是天然的图)。为什么分段?因为每段的影响因素不同(备餐取决于餐厅忙碌度,路况取决于交通),分开建模更准、更可解释、更容易定位误差来源。**能主动提出"把 ETA 拆成路段/阶段分别预测",是架构深度的体现。**
3. **诚实的难点:ETA 的魔鬼在实时性、长尾、和反馈回路**。①**实时路况是命门**:ETA 的准确性极度依赖**当前**交通状况(不是历史平均),所以要有一整套**流式特征管道**实时更新路况(接 21.4 流处理)——一场突发暴雨或事故能让 ETA 瞬间翻倍。②**长尾/罕见事件几乎无法预测**:模型能学好"正常情况"的 ETA,但事故、恶劣天气、大型活动这些罕见事件造成的极端延误,数据里样本极少、又极难预测——这也是为什么要预测分位数(把不确定性纳入)而非硬报一个点。③**分段误差累积**:分段建模虽好,但每段的误差会累积,长距离/多段的 ETA 更不准。④**反馈回路(微妙但重要)**:你报的 ETA 会**影响现实**——报得太长用户可能不下单(选别家)、报得太短司机可能超速或平台压力大;而且你的路径推荐会改变交通本身。你的预测在改变它要预测的世界,这让离线评估和因果分析都变复杂(接 19、24.1 的反馈回路主题)。⑤**冷启动**:新城市、新路段没有历史数据。**结论:设计 ETA 系统的核心不是"更准的回归器", 而是认识到 ETA 应预测分位数(P90 可靠承诺)而非点(因为误差不对称、迟到比早到伤), 用 pinball loss; 架构上分段/分路段建模再求和(而非端到端一个数), 依赖实时路况流式特征; 难点是长尾罕见事件、分段误差累积、ETA 影响现实的反馈回路——这是把回归、不确定性量化、实时系统、业务权衡融为一体的一道题。**

**English**:
1. **The most depth-revealing point of an ETA system: a regression problem needn't predict the "mean/median"**: most people hear "predict arrival time" and think ordinary regression (minimize MSE, predict the expected value). But ETA has a decisive business fact — **errors are asymmetric**: the pain of arriving 10 minutes late (the user waits anxiously at the door, the food gets cold, they miss a flight) far exceeds arriving 10 minutes early. This asymmetry directly overturns the default of "predict the median": our demo clearly shows the P50 (median) prediction is on-time only ~50% of the time — because by definition, the actual time exceeds the median half the time. The right approach is **predicting a high quantile (P90) as a "reliable promise,"** implemented via **quantile regression (pinball loss)** — which penalizes underestimation far more than overestimation, forcing the model to learn a buffered time. **"ETA should predict a distribution/quantile not a point, because errors are asymmetric" is this question's deepest insight and the divide between "can tune a regression model" and "understands the ETA business."**
2. **ETA's architectural wisdom: segment modeling, not one end-to-end number**: a novice might directly "input all features, output one ETA number." But a mature ETA system **decomposes the problem**: delivery ETA = order-accept wait + restaurant prep + courier pickup + on-road delivery, each modeled separately with its own features and uncertainty, then summed. The on-road segment further splits into **road segments** — cut the route into segments, predict each segment's travel time (based on that segment's real-time traffic and history), then add; more advanced uses **graph neural networks** (the road network is naturally a graph). Why segment? Because each segment's factors differ (prep depends on restaurant busyness, traffic depends on roads), so separate modeling is more accurate, interpretable, and easier for locating error sources. **Proactively proposing "split ETA into road segments/stages predicted separately" shows architectural depth.**
3. **Honest difficulty: ETA's devils are real-time, long tail, and feedback loops**. ① **Real-time traffic is the linchpin**: ETA accuracy depends heavily on **current** traffic (not historical average), so you need a full **streaming feature pipeline** updating traffic in real time (per 21.4 stream processing) — a sudden storm or accident can instantly double the ETA. ② **Long-tail/rare events are nearly unpredictable**: the model learns "normal" ETA well, but extreme delays from accidents, severe weather, or big events are rare in the data and very hard to predict — another reason to predict a quantile (incorporate uncertainty) rather than a hard point. ③ **Segment error accumulation**: segment modeling is good, but each segment's error accumulates, making long-distance/multi-segment ETAs less accurate. ④ **Feedback loop (subtle but important)**: your quoted ETA **affects reality** — too long and users may not order (choose elsewhere), too short and drivers may speed or the platform strains; and your route recommendations change traffic itself. Your prediction changes the world it predicts, complicating offline evaluation and causal analysis (per 19, 24.1's feedback-loop theme). ⑤ **Cold start**: new cities, new segments lack historical data. **Conclusion: designing an ETA system centers not on "a more accurate regressor" but on recognizing that ETA should predict a quantile (P90 reliable promise) not a point (because errors are asymmetric, late hurts more than early), using pinball loss; architecturally, model by segment/road-segment then sum (not one end-to-end number), relying on real-time traffic streaming features; difficulties are long-tail rare events, segment error accumulation, and the feedback loop of ETA affecting reality — a question fusing regression, uncertainty quantification, real-time systems, and business tradeoffs."*

> 💼 **实战视角 / Practical angle**
> **中文**:ETA 落地:①**预测分位数不是点**——分位数回归(pinball loss, GBDT/深度都支持), 输出 P80~P90 可靠承诺或区间, 按业务权衡准时率 vs 承诺吸引力;②**分段建模**:接单/备餐/取餐/配送各段单独预测再求和; 路上按路段/图网络(路网是图);③**实时特征**:流式路况(21.4)、天气、时段、餐厅排队、骑手状态——低延迟特征平台(22.9);④**模型**:GBDT 强基线 + 图网络(路网时空)+ 深度时空模型;⑤**监控**:MAE/MAPE + 分位数覆盖率/准时率, 关注长尾误差、分城市/时段分层;⑥**反馈回路**注意 ETA 影响下单和路径(用因果/实验评估真实影响)。**答题**:先抛出"误差不对称→预测分位数不是点", 再讲分段架构、实时路况、长尾和反馈回路。面试金句:*"ETA 是回归但关键是误差不对称(迟到比早到伤), 用分位数回归给 P90 可靠承诺(90%准时)而非 P50 中位数(一半迟到), 损失用 pinball; 架构分段建模(接单/备餐/配送、路上分路段或图网络)再求和; 实时路况流式特征是命门, 难点是长尾罕见事件、分段误差累积、ETA 改变现实的反馈回路; 指标看 MAE 和分位数覆盖率。"*
> **English**: ETA in practice: ① **predict quantiles not points** — quantile regression (pinball loss, supported by GBDT/deep), output a P80–P90 reliable promise or a range, trade off on-time rate vs promise attractiveness by business; ② **segment modeling**: order-accept/prep/pickup/delivery each predicted then summed; on-road by segments/graph nets (the road network is a graph); ③ **real-time features**: streaming traffic (21.4), weather, time-of-day, restaurant queue, courier state — low-latency feature store (22.9); ④ **models**: GBDT strong baseline + graph nets (road spatio-temporal) + deep spatio-temporal models; ⑤ **monitoring**: MAE/MAPE + quantile coverage/on-time rate, watch long-tail errors, stratify by city/time; ⑥ **feedback loop**: note ETA affects orders and routes (evaluate real impact with causal/experiments). **Answering**: first raise "asymmetric error → predict quantiles not points," then segment architecture, real-time traffic, long tail, and feedback loops. Interview line: *"ETA is regression but the key is asymmetric error (late hurts more than early), so use quantile regression for a P90 reliable promise (90% on time) not the P50 median (late half the time), with pinball loss; architecture is segment modeling (order-accept/prep/delivery, on-road by segments or graph nets) then summed; real-time traffic streaming features are the linchpin, difficulties are long-tail rare events, segment error accumulation, and the feedback loop of ETA changing reality; metrics are MAE and quantile coverage."*

---
### 小结 / Summary
- **中文**:ETA=回归, 但核心是预测分位数/可靠承诺而非点, 因为误差不对称(迟到比早到伤)→用分位数回归 P90(90%准时)而非 P50。
- **English**: ETA = regression, but the core is predicting a quantile/reliable promise not a point, because errors are asymmetric (late hurts more than early) → use quantile regression P90 (90% on time) not P50.
- **中文**:架构分段建模(接单/备餐/配送、路上分路段或图网络)再求和; 依赖实时路况流式特征。
- **English**: Architecture is segment modeling (order-accept/prep/delivery, on-road by segments or graph nets) then summed; relies on real-time traffic streaming features.
- **中文**:难点:长尾罕见事件、分段误差累积、ETA 改变现实的反馈回路; 指标 MAE/MAPE + 分位数覆盖率/准时率。
- **English**: Difficulties: long-tail rare events, segment error accumulation, the feedback loop of ETA changing reality; metrics MAE/MAPE + quantile coverage/on-time rate.
